# Customer Lifetime Value (LTV) Modeling

## Day 1 — LTV Data Preparation

In [2]:
import pandas as pd
df=pd.read_csv("../data/processed/telco_customer_churn_cleaned.csv")
df.head()

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,LTV
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,1936.30
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,107.70
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,1903.50
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,141.40


In [3]:
df.shape

(7043, 22)

In [4]:
df.columns

Index(['CustomerID', 'Gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'Tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'LTV'],
      dtype='str')

In [5]:
active_customers=df[df['Churn']=='No']
active_customers.shape

(5174, 22)

In [6]:
active_customers[["Tenure", "MonthlyCharges", "TotalCharges", "LTV"]].describe()

,Tenure,MonthlyCharges,TotalCharges,LTV
count,5174.000000,5174.000000,5174.000000,5174.000000
mean,37.569965,61.265124,2549.911442,2549.770883
std,24.113777,31.092648,2329.954215,2328.399619
min,0.000000,18.250000,0.000000,0.000000
25%,15.000000,25.100000,572.900000,574.562500
50%,38.000000,64.425000,1679.525000,1687.125000
75%,61.000000,88.400000,4262.850000,4244.812500
max,72.000000,118.750000,8672.450000,8550.000000


In [19]:
#DROPPED COLUMNS THAT ARE NOT USEFULL
X = active_customers.drop(
    ["LTV", "CustomerID", "Churn", "TotalCharges","Tenure","MonthlyCharges"],
    axis=1
)
y = active_customers["LTV"]
X.shape

(5174, 16)

In [20]:
X.columns

Index(['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [21]:
X.dtypes

Gender              str
SeniorCitizen       str
Partner             str
Dependents          str
PhoneService        str
MultipleLines       str
InternetService     str
OnlineSecurity      str
OnlineBackup        str
DeviceProtection    str
TechSupport         str
StreamingTV         str
StreamingMovies     str
Contract            str
PaperlessBilling    str
PaymentMethod       str
dtype: object

### Encode categorical features

In [22]:
#onehotencoder and columntransformer helps to keep the numerical column as the are  and automatically convert  strings to numericals (0/1)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X.select_dtypes(include=["str"]).columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns

print("Categorical:", list(categorical_features))
print("Numeric:", list(numeric_features))


Categorical: ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numeric: []


### Train/Test Split

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((4139, 16), (1035, 16))

In [24]:
y_train.shape,y_test.shape

((4139,), (1035,))

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
         OneHotEncoder(
             drop="first",
             handle_unknown="ignore"
         ),
         categorical_features),

        ("num",
         "passthrough",
         numeric_features)
    ]
)

In [26]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [27]:
X_train_processed.shape, X_test_processed.shape

((4139, 27), (1035, 27))